In [1]:
import pandas as pd

In [2]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

# Save as csv file
df = pd.DataFrame(data)
df.to_csv('data/csv_files/products.csv',index=False)

In [4]:
# Save as excel file with multiple sheets
with pd.ExcelWriter('data/excel_files/inventory.xlsx') as writer:
    df.to_excel(writer, sheet_name='Products', index=False)

    # Add another sheet
    summary_data = {
        'Category': ['Electronics', 'Accessories'],
        'Total_Items': [3, 2],
        'Total_Value': [1389.97, 109.98]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)


# CSV Processing

In [6]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

c:\code2\Natural Language Processing Projects\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Method 1: CSVLoader - Each row becomes a document
csv_loader = CSVLoader(
    file_path='data/csv_files/products.csv',
    encoding='utf-8',
    csv_args={
        'delimiter':',',
        'quotechar':'"'
    }
)

csv_docs = csv_loader.load()
print(f"Loaded {len(csv_docs)} documents (one per row)")
for i, doc in enumerate(csv_docs):
    print(f"Document {i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")
    print()

Loaded 5 documents (one per row)
Document 1:
Metadata: {'source': 'data/csv_files/products.csv', 'row': 0}
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD

Document 2:
Metadata: {'source': 'data/csv_files/products.csv', 'row': 1}
Content: Product: Mouse
Category: Accessories
Price: 29.99
Stock: 200
Description: Wireless optical mouse with ergonomic design

Document 3:
Metadata: {'source': 'data/csv_files/products.csv', 'row': 2}
Content: Product: Keyboard
Category: Accessories
Price: 79.99
Stock: 150
Description: Mechanical keyboard with RGB backlighting

Document 4:
Metadata: {'source': 'data/csv_files/products.csv', 'row': 3}
Content: Product: Monitor
Category: Electronics
Price: 299.99
Stock: 75
Description: 27-inch 4K monitor with HDR support

Document 5:
Metadata: {'source': 'data/csv_files/products.csv', 'row': 4}
Content: Product: Webcam
Category: Electronics
Price: 89.99
Stock: 100
Descripti

In [13]:
from typing import List
from langchain_core.documents import Document

# Method 2: Custom CSV processing for better context
def custom_csv_processing(filepath:str)->List[Document]:
        df = pd.read_csv(filepath)
        documents = []

        for i,row in df.iterrows():
            content = f"""Product Information:
            Name: {row['Product']}
            Category: {row['Category']}
            Price: {row['Price']}
            Stock: {row['Stock']} units
            Description: {row['Description']}"""

            doc = Document(
                  page_content=content,
                  metadata={
                    'source': filepath,
                    'row_index': i,
                    'relate to': 'another file',
                    'data_type': 'product_info'
                  }
            )

            documents.append(doc)
        
        return documents

In [18]:
customed_csv_docs = custom_csv_processing('data/csv_files/products.csv')

print(f"Loaded {len(customed_csv_docs)} documents (one per row)")
for i, doc in enumerate(customed_csv_docs):
    print(f"Document {i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")
    print()

Loaded 5 documents (one per row)
Document 1:
Metadata: {'source': 'data/csv_files/products.csv', 'row_index': 0, 'relate to': 'another file', 'data_type': 'product_info'}
Content: Product Information:
            Name: Laptop
            Category: Electronics
            Price: 999.99
            Stock: 50 units
            Description: High-performance laptop with 16GB RAM and 512GB SSD

Document 2:
Metadata: {'source': 'data/csv_files/products.csv', 'row_index': 1, 'relate to': 'another file', 'data_type': 'product_info'}
Content: Product Information:
            Name: Mouse
            Category: Accessories
            Price: 29.99
            Stock: 200 units
            Description: Wireless optical mouse with ergonomic design

Document 3:
Metadata: {'source': 'data/csv_files/products.csv', 'row_index': 2, 'relate to': 'another file', 'data_type': 'product_info'}
Content: Product Information:
            Name: Keyboard
            Category: Accessories
            Price: 79.99
   

# Excel Processing

In [20]:
# Method 1: UnstructuredExcelLoader
from langchain_community.document_loaders import UnstructuredExcelLoader
excel_loader = UnstructuredExcelLoader(
    'data/excel_files/inventory.xlsx',
    mode='elements'
)

unstructured_docs = excel_loader.load()

print(f"Loaded {len(unstructured_docs)} document(s)")
for i, doc in enumerate(unstructured_docs):
    print(f"Document {i+1}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")
    print()

Loaded 2 document(s)
Document 1
Metadata: {'source': 'data/excel_files/inventory.xlsx', 'file_directory': 'data/excel_files', 'filename': 'inventory.xlsx', 'last_modified': '2025-10-23T12:20:20', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K monitor with HDR support</td></tr><tr><td>Webcam</td><td>Electronics</td><td>89.99</td><td>100</td><td>1080p webcam with noise cancellation</td></tr></table>', 'languages': ['eng'], 'filetype'

In [21]:
# Method 2: Using pandas for full control
def process_excel_with_pandas(filepath:str)->List[Document]:
    """Process Excel with sheet awareness"""
    documents = []

    # Read all sheets
    excel_file = pd.ExcelFile(filepath)
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath,sheet_name=sheet_name)

        # Create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {','.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)

    return documents 

In [22]:
excel_docs = process_excel_with_pandas('data/excel_files/inventory.xlsx')

print(f"Loaded {len(excel_docs)} document(s)")
for i, doc in enumerate(excel_docs):
    print(f"Document {i+1}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}")
    print()

Loaded 2 document(s)
Document 1
Metadata: {'source': 'data/excel_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}
Content: Sheet: Products
Columns: Product,Category,Price,Stock,Description
Rows: 5

 Product    Category  Price  Stock                                         Description
  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD
   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design
Keyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting
 Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support
  Webcam Electronics  89.99    100                1080p webcam with noise cancellation

Document 2
Metadata: {'source': 'data/excel_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}
Content: Sheet: Summary
Columns: Category,Total_Items,Total_Value